# SentinelIQ — 02 Isolation Forest
Train and evaluate Isolation Forest on metrics and network flow data.

In [ ]:
!git clone https://github.com/YOUR_USERNAME/SentinelIQ.git 2>/dev/null || echo "Already cloned"
%cd /kaggle/working/SentinelIQ
import sys
sys.path.insert(0, '/kaggle/working/SentinelIQ')
!pip install pyyaml joblib scikit-learn -q


In [ ]:
!python data/simulated/pipeline.py --duration 120 --anomaly-rate 0.08


In [ ]:
import json
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

from ml.models.isolation_forest import SentinelIsolationForest

sns.set_theme(style="darkgrid")

with open('configs/model_config.yaml') as f:
    cfg = yaml.safe_load(f)

def load_jsonl(path):
    return pd.DataFrame([json.loads(l) for l in open(path)])


## Train on Metrics

In [ ]:
metrics_df = load_jsonl('data/simulated/metrics.jsonl')
feature_cols = cfg['isolation_forest']['features']
y = metrics_df['is_anomaly'].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    metrics_df, y, test_size=0.2, random_state=42, stratify=y)

X_train_normal = X_train[y_train == 0]
print(f"Training on {len(X_train_normal)} normal samples")
print(f"Test set: {len(X_test)} samples | {y_test.sum()} anomalies")


In [ ]:
if_metrics = SentinelIsolationForest(config=cfg['isolation_forest'], feature_cols=feature_cols)
if_metrics.fit(X_train_normal)


In [ ]:
metrics_results = if_metrics.evaluate(X_test, y_test)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC Curve
scores = if_metrics.score(X_test)
RocCurveDisplay.from_predictions(y_test, scores, ax=axes[0], name='Isolation Forest')
axes[0].set_title(f"ROC Curve (AUC={metrics_results['roc_auc']})")

# Precision-Recall
PrecisionRecallDisplay.from_predictions(y_test, scores, ax=axes[1], name='Isolation Forest')
axes[1].set_title(f"Precision-Recall (AP={metrics_results['avg_precision']})")

# Score distributions
normal_scores = scores[y_test == 0]
anomaly_scores = scores[y_test == 1]
axes[2].hist(normal_scores, bins=30, alpha=0.6, color='#2ecc71', label='Normal')
axes[2].hist(anomaly_scores, bins=30, alpha=0.6, color='#e74c3c', label='Anomaly')
axes[2].axvline(if_metrics.threshold, color='navy', linestyle='--', label=f'Threshold={if_metrics.threshold:.3f}')
axes[2].set_title('Anomaly Score Distribution')
axes[2].legend()

plt.suptitle('Isolation Forest — Metrics', fontsize=14)
plt.tight_layout()
plt.show()


## Train on Network Flows

In [ ]:
network_df = load_jsonl('data/simulated/network.jsonl')
net_features = cfg['network']['features']
y_net = network_df['is_anomaly'].astype(int).values

X_train_net, X_test_net, y_train_net, y_test_net = train_test_split(
    network_df, y_net, test_size=0.2, random_state=42, stratify=y_net)

X_train_net_normal = X_train_net[y_train_net == 0]

network_cfg = {**cfg['isolation_forest'], **cfg['network']}
if_network = SentinelIsolationForest(config=network_cfg, feature_cols=net_features)
if_network.fit(X_train_net_normal)
network_results = if_network.evaluate(X_test_net, y_test_net)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

net_scores = if_network.score(X_test_net)
RocCurveDisplay.from_predictions(y_test_net, net_scores, ax=axes[0], name='IF Network')
axes[0].set_title(f"ROC Curve (AUC={network_results['roc_auc']})")

PrecisionRecallDisplay.from_predictions(y_test_net, net_scores, ax=axes[1], name='IF Network')
axes[1].set_title(f"Precision-Recall (AP={network_results['avg_precision']})")

axes[2].hist(net_scores[y_test_net==0], bins=30, alpha=0.6, color='#2ecc71', label='Normal')
axes[2].hist(net_scores[y_test_net==1], bins=30, alpha=0.6, color='#e74c3c', label='Anomaly')
axes[2].axvline(if_network.threshold, color='navy', linestyle='--', label=f'Threshold')
axes[2].set_title('Score Distribution — Network')
axes[2].legend()

plt.tight_layout()
plt.show()


## Save Models

In [ ]:
import os
os.makedirs('ml/saved_models', exist_ok=True)
if_metrics.save('ml/saved_models', name='isolation_forest_metrics')
if_network.save('ml/saved_models', name='isolation_forest_network')
print("Models saved.")


## Results Summary

In [ ]:
summary = pd.DataFrame([
    {'Model': 'IF Metrics', **{k:v for k,v in metrics_results.items() if k != 'confusion_matrix'}},
    {'Model': 'IF Network', **{k:v for k,v in network_results.items() if k != 'confusion_matrix'}},
])
print(summary[['Model','roc_auc','avg_precision','precision','recall','f1','accuracy']].to_string(index=False))
